# Supervisor Results Extract

Extract canonical climatology tables from `data/processed/p_favorable_spb.nc` for Sections 4.4 and 4.5.

In [1]:
import xarray as xr
from pathlib import Path
root = Path("..") if Path.cwd().name == "notebooks" else Path(".")
ds = xr.open_dataset(root / "data/processed/p_favorable_spb.nc")
seasons = ["DJF", "MAM", "JJA", "SON"]
periods = ["day", "evening", "night"]
variables = ["p_favorable", "p_favorable_wind", "p_favorable_thermal"]
ds = ds.sel(season=seasons, period=periods)

def md(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    return "\n".join(lines + ["| " + " | ".join(map(str, row)) + " |" for row in rows])

def means(cell=None):
    out = {}
    for name in variables:
        da = ds[name]
        if cell is None:
            vals = da.weighted(ds["n_samples"]).mean("cell")
            vals = vals.mean("sector") if "sector" in vals.dims else vals
        else:
            vals = da.isel(cell=cell)
            vals = vals.mean("sector") if "sector" in vals.dims else vals
        out[name] = vals
    return out

def by_period(arrs):
    rows = []
    for season in seasons:
        for name in variables:
            vals = arrs[name].sel(season=season)
            rows.append([season, name, *[f"{float(vals.sel(period=p)):.3f}" for p in periods]])
    return md(["season", "variable", *periods], rows)

domain, cell16 = means(), means(16)
contrast = []
for name, da in domain.items():
    ser = da.to_series(); hi, lo = ser.idxmax(), ser.idxmin()
    contrast.append([name, f"{float(ser.loc[hi] / ser.loc[lo]):.2f}", f"{hi[0]}-{hi[1]}", f"{ser.loc[hi]:.3f}", f"{lo[0]}-{lo[1]}", f"{ser.loc[lo]:.3f}"])
c = ds.isel(cell=16).sel(season="DJF", period="night")
thermal = float(c["p_favorable_thermal"])
sector_rows = [[f"{float(az):.0f}", f"{float(c['p_favorable_wind'].isel(sector=i)):.3f}", f"{thermal:.3f}", f"{float(c['p_favorable'].isel(sector=i)):.3f}"] for i, az in enumerate(ds["sector_az"].values)]
sections = [
    "## Domain mean by (season, period)\n\n" + by_period(domain),
    "## Cell 16 by (season, period)\n\n" + by_period(cell16),
    "## Contrast ratios\n\n" + md(["variable", "ratio", "max", "max value", "min", "min value"], contrast),
    "## Cell 16, DJF-night, per sector\n\n" + md(["sector_az_iso_deg", "p_favorable_wind", "p_favorable_thermal", "p_favorable"], sector_rows),
]
out = root / "docs/notes/supervisor_results_tables.md"
out.write_text("\n\n".join(sections) + "\n", encoding="utf-8")
print(out.read_text(encoding="utf-8"))


/Users/meteof/Projects/noise-meteo-spb/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Domain mean by (season, period)

| season | variable | day | evening | night |
| --- | --- | --- | --- | --- |
| DJF | p_favorable | 0.603 | 0.646 | 0.675 |
| DJF | p_favorable_wind | 0.287 | 0.294 | 0.287 |
| DJF | p_favorable_thermal | 0.433 | 0.484 | 0.530 |
| MAM | p_favorable | 0.434 | 0.600 | 0.805 |
| MAM | p_favorable_wind | 0.262 | 0.243 | 0.243 |
| MAM | p_favorable_thermal | 0.233 | 0.457 | 0.717 |
| JJA | p_favorable | 0.360 | 0.575 | 0.868 |
| JJA | p_favorable_wind | 0.249 | 0.215 | 0.215 |
| JJA | p_favorable_thermal | 0.145 | 0.443 | 0.810 |
| SON | p_favorable | 0.474 | 0.636 | 0.694 |
| SON | p_favorable_wind | 0.283 | 0.270 | 0.265 |
| SON | p_favorable_thermal | 0.260 | 0.479 | 0.557 |

## Cell 16 by (season, period)

| season | variable | day | evening | night |
| --- | --- | --- | --- | --- |
| DJF | p_favorable | 0.586 | 0.601 | 0.618 |
| DJF | p_favorable_wind | 0.336 | 0.342 | 0.338 |
| DJF | p_favorable_thermal | 0.377 | 0.391 | 0.422 |
| MAM | p_favorable 